# Theseus 教程（中文翻译版）

- 原始英文版：`03_custom_cost_functions.ipynb`
- 说明：本文件为自动翻译版本（保留代码不翻译，清空输出以减小体积）。如遇术语不一致，可优先参考英文原文。


<h1>创建自定义成本函数</h1>

在本教程中，我们将展示如何创建应用程序可能需要的自定义成本函数。虽然我们总是可以通过简单地编写错误函数来使用 `AutoDiffCostFunction`，但对于计算密集型应用程序来说，派生新的 `CostFunction` 子类并使用封闭形式雅可比行列式通常更有效。

我们将在本教程中展示如何编写自定义 `VectorDifference` 成本函数。该成本函数提供两个 `Vector` 之间的差异作为误差。

注意：`VectorDifference` 是Theseus 库中已提供的 `Difference` 成本函数的简化版本，如教程 0 所示。`Difference` 可以在任何 LieGroup 上使用，而 `VectorDifference` 只能在向量上使用。

<h2>初始化</h2>

任何 `CostFunction` 子类都应使用 `CostWeight` 以及计算成本函数所需的所有参数进行初始化。在此示例中，我们为 `VectorDifference` 设置 `__init__` 函数，要求输入我们希望计算其差异的两个 `Vector`：要优化的 `Vector`、`var` 和`Vector`为比较参考，`target`。

另外，`__init__`函数还需要注册优化变量和所有辅助变量。在此示例中，优化变量 `var` 注册到 `register_optim_vars`。评估成本所需的另一个输入 `target` 注册到 `register_aux_vars`。这是非线性优化器正常工作所必需的：这些函数将优化和辅助变量注册到内部列表中，然后相关的 `Objective` 可以轻松地使用它们来添加它们，确保没有名称冲突，并用新值更新它们。

`CostWeight` 用于对误差和雅可比进行加权，并且每个 `CostFunction` 子类都需要它（误差和雅可比加权函数是从父 `CostFunction` 类继承的。）

In [ ]:
from typing import List, Optional, Tuple
import theseus as th

class VectorDifference(th.CostFunction):
    def __init__(
        self,
        cost_weight: th.CostWeight,
        var: th.Vector,
        target: th.Vector,
        name: Optional[str] = None,
    ):
        super().__init__(cost_weight, name=name) 

        # add checks to ensure the input arguments are of the same class and dof:
        if not isinstance(var, target.__class__):
            raise ValueError(
                "Variable for the VectorDifference inconsistent with the given target."
            )
        if not var.dof() == target.dof():
            raise ValueError(
                "Variable and target in the VectorDifference must have identical dof."
            )

        self.var = var
        self.target = target

        # register variable and target
        self.register_optim_vars(["var"])
        self.register_aux_vars(["target"])

<h2>实现抽象函数</h2>

接下来，我们需要实现`CostFunction`的抽象函数：`dim`、`error`、`jacobians`和`_copy_impl`：
- `dim`：返回错误的自由度（`dof`）；在本例中，这是优化变量 `var` 的 `dof`
- `error`：返回向量的差异，即 `var` - `target`
- `jacobian`：返回相对于 `var` 的错误的雅可比行列式
- `_copy_impl`：创建内部类成员的深层副本

我们在下面说明这些（再次包括上面的 `__init__` 函数，因此该类已完全定义。）

In [ ]:
import torch 

class VectorDifference(th.CostFunction):
    def __init__(
        self,
        cost_weight: th.CostWeight,
        var: th.Vector,
        target: th.Vector,
        name: Optional[str] = None,
    ):
        super().__init__(cost_weight, name=name) 
        self.var = var
        self.target = target
        # to improve readability, we have skipped the data checks from code block above
        self.register_optim_vars(["var"])
        self.register_aux_vars(["target"])

    def error(self) -> torch.Tensor:
        return (self.var - self.target).tensor

    def jacobians(self) -> Tuple[List[torch.Tensor], torch.Tensor]:
        return [
            # jacobian of error function wrt var is identity matrix I
            torch.eye(self.dim(), dtype=self.var.dtype)  
            # repeat jacobian across each element in the batch
            .repeat(self.var.shape[0], 1, 1)  
            # send to variable device
            .to(self.var.device)  
        ], self.error()

    def dim(self) -> int:
        return self.var.dof()

    def _copy_impl(self, new_name: Optional[str] = None) -> "VectorDifference":
        return VectorDifference(  # type: ignore
            self.var.copy(), self.weight.copy(), self.target.copy(), name=new_name
        )

<h2> 方便</h2>

我们现在证明 `VectorDifference` 成本函数按预期工作。

为此，我们在一对 `Vector` <i>a_i</i> 和 <i>b_i</i> 上创建一组 `VectorDifference` 成本函数，并将它们添加到 `Objective`。然后，我们为 `VectorDifference` 成本函数的每个 `Vector` <i>a_i</i> 和 <i>b_i</i> 创建数据，并为 `update` 和 `Objective` 创建数据。下面的代码片段显示 `Objective` 错误已正确计算。

我们在这里使用 `ScaleCostWeight` 作为输入 `CostWeight`：这是一个标量实值 `CostWeight`，用于对 `CostFunction` 进行加权；为简单起见，在此示例中我们使用固定值 1。

In [ ]:
cost_weight = th.ScaleCostWeight(1.0)

# construct cost functions and add to objective
objective = th.Objective()
num_test_fns = 10
for i in range(num_test_fns):
    a = th.Vector(2, name=f"a_{i}")
    b = th.Vector(2, name=f"b_{i}")
    cost_fn = VectorDifference(cost_weight, a, b)
    objective.add(cost_fn)
    
# create data for adding to the objective
theseus_inputs = {}
for i in range(num_test_fns):
    # each pair of var/target has a difference of [1, 1]
    theseus_inputs.update({f"a_{i}": torch.ones((1,2)), f"b_{i}": 2 * torch.ones((1,2))})

objective.update(theseus_inputs)
# sum of squares of errors [1, 1] for 10 cost fns: the result should be 20
error_sq = objective.error_metric()
print(f"Sample error squared norm: {error_sq.item()}")